# 第05课：Embedding与向量检索

上一课解决了“聊天记录怎么带、回答怎么收”；这一课解决“回答前，到哪里找相关资料”。主线项目要查询指定资料，所以先做一个**只返回候选资料、不生成回答**的小检索器。

本课目标：①分清文本与向量、维度与兼容空间；②算余弦分数；③取Top-K并允许门槛筛选后没有结果。三组分别学习，不必一次读完。示例是博物馆、课堂练习是校园、作业是客服。

**能力边界**：本课全程离线。手工指定的教学向量不是模型Embedding，不能证明真实语义理解或任意中文检索效果。只练清楚生成端之后的检索计算。无密钥、无API、无模型下载、无向量数据库。真实文本编码的模型选择与接线不作为本课作业。

入口：[3个HTML微课](00_学习入口.md)；[阶段使命](../MISSION.md)。第四课课堂练习1副本分支仍待补正，本课不改旧答案。

## 运行本课前：安装与检查

Python 3.10及以上。计算只用标准库；pytest只用于课后测试，不是Embedding模型。终端先进入本课目录（从仓库根目录执行；PowerShell也适用）：

```powershell
cd "03_大模型应用开发/05_Embedding与向量检索"
python -m pip install -r requirements.txt
python -m pip show pytest
python -m pytest --collect-only -q test_vector_search.py
```

python是所选解释器；-m pip用它运行安装工具；install安装；-r读取依赖清单。最后一条只收集20项测试，不执行作业。

Notebook也可新建临时格运行`%pip install -r requirements.txt`；%pip面向当前内核，终端python可能是另一环境。可先用下一格确认目录；安装后若导入仍失败，重启内核再顺序运行。不要把安装命令写进作业函数。

In [26]:
from pathlib import Path
# Path是标准库pathlib中的路径类；Path.cwd()返回当前工作目录的Path对象。
lesson_dir = Path.cwd()
if not (lesson_dir / "support_vectors.json").is_file():
    lesson_dir = lesson_dir / "03_大模型应用开发" / "05_Embedding与向量检索"
assert (lesson_dir / "support_vectors.json").is_file(), "请从本课目录或仓库根目录运行Notebook"
print("已定位本课资料；没有联网。")

已定位本课资料；没有联网。


## 0. 先看数据走到哪里

Embedding（嵌入）在本课指：把文本用模型编码为一串用于比较的数字；这一串数字叫向量。检索指从已有资料中找候选，不是让模型写答案。

```text
建库时：资料文本 → 文档编码 → 保存原文、来源和向量
                         ↑ 本课用手工教学向量代替这一步
查询时：问题文本 → 查询编码 → 查询向量
                               ↓
T1 读取已有向量 → T2 逐条算分 → T3 降序取Top-K → T4 门槛筛选
                                                     ↓
                                             候选原文和来源
下课：把候选原文放进messages → 聊天模型生成回答
```

建库不是每次查询都重做；资料或编码配置改变时才需更新相应向量。不要把“检索结果”当成聊天模型已经作答，也不要把向量当原文发给聊天模型代替资料。

本课的编码输入已准备好，你实现T2～T4。没有框架偷偷帮你排序，数据处理都由Python代码完成。

原始资料：[Google的向量表示说明](https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space)、[Sentence Transformers语义检索流程](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)。

## 1. 文本和向量不是同一种数据

向量可以先理解为**有顺序的数字列表**。`[3.0, 4.0]`有两个分量（每个位置上的数），所以是二维；`[0.2, -0.1, 0.8]`是三维。不是“两条文本”就叫二维，也不是字符越多，维度就越大。

真实Embedding模型学习如何表示文本，模型配置通常决定维度。相近表达可能映射到接近方向，但每个坐标通常不能直接解释为“密码程度”。下方数字是教师为了计算方便手工指定的坐标，不是神经网络结果。

先看一个资料记录：

In [2]:
museum_record = {
    "id": "m1",
    "text": "入馆前需要线上预约。",
    "source": "参观指南#预约",
    "vector": [3.0, 4.0],  # 教师手工教学向量
}
print(museum_record["text"])
print(museum_record["vector"])
print("维度：", len(museum_record["vector"]))
print("换一段更长的文本，不会自动改变已存的向量：", museum_record["vector"])

入馆前需要线上预约。
[3.0, 4.0]
维度： 2
换一段更长的文本，不会自动改变已存的向量： [3.0, 4.0]


### 1.1 谁提供这些字段？

以下id/text/source/vector/score都是应用的数据字段，不是Python关键字；本课没有HTTP请求，也没有服务器自动生成字段。

| 字段 | 类型 | 数据由谁提供 | 用途 |
|---|---|---|---|
| id | str | 应用准备资料时指定 | 区分每条资料 |
| text | str | 资料作者提供原文 | 命中后把原文交给后续流程 |
| source | str | 应用记录文档名和位置 | 找回原文位置，不是模型编造的引用 |
| vector | list[float] | 本课由教师指定；真实应用由编码模型输出 | 打分使用，不是回答正文 |
| score | float | 检索代码计算 | 排序与门槛筛选，不是概率 |

原文、来源和向量属于同一条资料，要保持配对。查询向量是另一份数字列表，不要拿问题字符串直接和列表相乘。

你上一课见过result的text与消息的content：本课资料正文统一叫text。把候选资料放进messages，是下一课才做的事。

### 1.2 维度一样，还不一定能比较

比较时要求：查询和资料来自**兼容的同一向量空间**，并使用模型规定的查询/文档编码方式。不能把模型甲的查询向量和模型乙的资料向量混在一起，即使它们都是三维。

类比两张地图：同样是两个数字，不代表经纬度和教室平面坐标可以直接比较。换编码模型后通常需要重建资料向量。真实模型可能要求query/document不同前缀；不要擅自把一边编码设置改掉。

最小正常：同一套教学坐标下比较[1,0]与[0.8,0.6]。变化：两段不同长度文本可以得到同维度向量。反例：模型甲的[1,0]与模型乙的[1,0]，数值碰巧相同不证明可比较。

**先预测，再运行检测**：长度为3的向量有几维？两个模型输出相同维度是否足以保证空间兼容？先填下一格：第二项填True或False。来源：[官方查询/文档编码说明](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)。

In [3]:
predict_dimensions = 3  # TODO：整数
predict_same_space = False  # TODO：True或False

In [4]:
try:
    _check_case = "维度与兼容空间预测"
    assert type(globals().get("predict_dimensions")) is int, "先填写predict_dimensions整数"
    assert predict_dimensions == 3, "三个坐标分量对应三维，不是按文本字数计"
    assert type(globals().get("predict_same_space")) is bool, "先填写predict_same_space布尔值"
    assert predict_same_space is False, "维度相同只是必要条件；还要核对空间和编码配置兼容"
    print("✅ 预测1通过")
except AssertionError as _check_error:
    print("❌ 预测1：", str(_check_error))
except Exception:
    print("❌ 预测1：", _check_case if "_check_case" in globals() else "请先运行本题答题单元", "；检查返回类型、字段及计算规则，不要改检测代码。")

✅ 预测1通过


<details>
<summary>独立完成并检测后，再查看本题参考</summary>

predict_dimensions应为3，predict_same_space应为False。同维度不证明同空间。

</details>

## 2. 余弦相似度：先看方向，再看公式

把二维向量想成从原点出发的箭头。[1,0]向右，[0,1]向上，[2,0]仍向右，只是更长。余弦相似度比较两个非零向量的方向：同向1、垂直0、反向-1。越大表示在当前表示下越相近，不是“回答有多少概率正确”。

[微课2](../lessons/0005-cosine-direction.html)可以切换方向。先用数字走一遍，不需要三角函数推导。

需要两种计算：

1. **点积**：对应位置相乘再相加。[3,4]与[6,8]的点积=3×6+4×8=50。
2. **长度（范数）**：各分量平方求和再开平方。[3,4]长度=√(9+16)=5。

`余弦 = 点积 / (第一个向量长度 × 第二个向量长度)`。这个例子为50/(5×10)=1，而不是50。

原始公式：[scikit-learn余弦定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)。只引用公式，不需要安装scikit-learn。

### 2.1 对应到Python

`sqrt(x)`来自标准库math：位置参数x为非负数字，返回平方根float；例如sqrt(25)=5.0。

`zip(a,b)`是内置函数，接收两个可遍历对象，逐项提供(a的分量,b的分量)；for里的x、y分别接住这两个数。注意：默认zip遇到较短一方结束就停止，不会自动报告维度不同。

`sum(数字序列)`是内置求和函数。`x*x`为平方；本例先写普通循环，不同时引入公式和复杂推导式。接下来每行都在本地执行，不是API字段转换。

In [5]:
from math import sqrt
museum_query_vector = [3.0, 4.0]
museum_document_vector = [6.0, 8.0]
dot_product = 0.0
for x, y in zip(museum_query_vector, museum_document_vector):
    dot_product += x * y
query_squares = sum(x * x for x in museum_query_vector)
document_squares = sum(x * x for x in museum_document_vector)
query_norm = sqrt(query_squares)
document_norm = sqrt(document_squares)
similarity = dot_product / (query_norm * document_norm)
print("点积：", dot_product, "；长度：", query_norm, document_norm)
print("余弦：", similarity)

点积： 50.0 ；长度： 5.0 10.0
余弦： 1.0


执行顺序：先配对相乘累计50 → 分别算出长度5和10 → 除以长度乘积得到1。`sum(x*x for x in ...)`是把逐项平方结果交给sum累加；这里没有创建模型，也没有读取文本。

为什么不直接用点积？把[6,8]扩大成[60,80]后，点积增到500，但方向没变，余弦仍是1。只有两边向量都已被处理成长度1时，点积才等于余弦；本课不保证输入已处理成长度1。

### 2.2 两个真正影响计算的错误

- [1,0]与[1,0,9]不能逐维对齐。zip会静默忽略9，所以必须在公式前明确维度要求。
- [0,0]没有方向，两边长度乘积为0，无法用这个公式除。不能当成“特别相似”。本课函数约定对此抛ValueError；某些库把这种情形定义为0，是库的工程约定，不是本课约定。

下一格把拒绝原因显示出来，不让你靠除零错误猜。

In [6]:
for left, right in (([1.0, 0.0], [1.0, 0.0, 9.0]), ([0.0, 0.0], [1.0, 0.0])):
    if len(left) != len(right):
        print("不能计算：向量维度不同")
    elif sum(x*x for x in left) == 0 or sum(x*x for x in right) == 0:
        print("不能计算：零向量没有方向")
print("zip默认截断的反例：", list(zip([1, 0], [1, 0, 9])))

不能计算：向量维度不同
不能计算：零向量没有方向
zip默认截断的反例： [(1, 1), (0, 0)]


**先预测，再运行**：在同一空间，[1,0]与[0.8,0.6]的两边长度都是1。分数是多少？把第二个向量放大为[8,6]后，分数是多少？先填两个float，再运行检测。

In [8]:
predict_score = 0.8  # TODO：float
predict_scaled_score = 0.8  # TODO：float

In [9]:
try:
    _check_case = "方向缩放预测"
    from math import isclose as _check_close
    # isclose比较浮点数是否足够接近；abs_tol为允许的绝对误差。
    for _check_name in ("predict_score", "predict_scaled_score"):
        assert type(globals().get(_check_name)) in (float, int), f"先填写{_check_name}数值"
        assert _check_close(globals()[_check_name], 0.8, abs_tol=1e-9), f"{_check_name}应按点积除两长度计算，正比例放大不改方向"
    print("✅ 预测2通过")
except AssertionError as _check_error:
    print("❌ 预测2：", str(_check_error))
except Exception:
    print("❌ 预测2：", _check_case if "_check_case" in globals() else "请先运行本题答题单元", "；检查返回类型、字段及计算规则，不要改检测代码。")

✅ 预测2通过


<details>
<summary>独立完成并检测后，再查看本题参考</summary>

两个预测都是0.8。放大后点积为8，向量长度乘积为10；8/10仍为0.8。

</details>

### 2.3 课堂练习1：校园资料的方向分数（5～8分钟）

目标：把点积、长度和余弦关系写成一个函数。实现`campus_cosine(a,b)->float`。a、b都是list[float]，代表校园问题与资料的教学向量；保证非空、同维度、非零。不用检查任意类型，不调用模型。

例：[1.0,0.0]和[3.0,4.0]应返回约0.6；[2.0,0.0]和[-5.0,0.0]应返回约-1.0。适用于二维以上，不只写死两个分量。

只在下一格pass处写实现。先运行答题格，再运行紧下方检测；pass是未作答占位，不是答案。检测内部用isclose容忍浮点舍入误差。

In [14]:
from math import sqrt

def campus_cosine(a: list[float], b: list[float]) -> float:
    dot_product = sum(x * y for x, y in zip(a, b))
    a_length = sqrt(sum(x * x for x in a))
    b_length = sqrt(sum(y * y for y in b))

    return dot_product / (a_length * b_length)

In [15]:
try:
    _check_case = "课堂练习1尚未定义campus_cosine"
    assert callable(globals().get("campus_cosine")), "先运行campus_cosine答题格"
    from math import isclose as _check_close
    for _check_left, _check_right, _check_expected, _check_case in (
        ([1.0,0.0], [3.0,4.0], 0.6, "非单位向量应除以长度"),
        ([2.0,0.0], [7.0,0.0], 1.0, "同方向不同长度"),
        ([1.0,0.0], [0.0,1.0], 0.0, "垂直方向"),
        ([2.0,0.0], [-5.0,0.0], -1.0, "反向方向"),
        ([0.0,0.0,2.0], [0.0,3.0,4.0], 0.8, "三维应使用所有分量"),
    ):
        _check_actual = campus_cosine(_check_left, _check_right)
        assert type(_check_actual) in (float, int), f"{_check_case}：需要返回分数，不是None或列表"
        assert _check_close(_check_actual, _check_expected, abs_tol=1e-9), f"{_check_case}：分数不符合余弦公式"
    print("✅ 课堂练习1通过")
except AssertionError as _check_error:
    print("❌ 课堂练习1：", str(_check_error))
except Exception:
    print("❌ 课堂练习1：", _check_case if "_check_case" in globals() else "请先运行本题答题单元", "；检查返回类型、字段及计算规则，不要改检测代码。")

✅ 课堂练习1通过


<details>
<summary>独立完成并检测后，再查看本题参考</summary>

```python
from math import sqrt

def campus_cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    a_length = sqrt(sum(x*x for x in a))
    b_length = sqrt(sum(y*y for y in b))
    return dot / (a_length*b_length)
```

此课堂题保证有效向量；课后任务1另外要求区分维度不同和零向量。先解释为何分母存在，再迁移，不只是改函数名。

</details>

接下来示例也需要计算分数。下面是**教师提供的有效输入示例函数**demo_cosine，只服务博物馆演示，不是你的课后作业实现。参数保证为两个有效的非零同维度列表，返回float；它不处理无效输入。课后任务1还需要明确拒绝维度不同和零向量，不能直接把这个示例当成完整作业答案。函数名不是库函数。

In [23]:
def demo_cosine(a, b):
    a_length = sqrt(sum(x*x for x in a))
    b_length = sqrt(sum(x*x for x in b))
    return sum(x*y for x,y in zip(a,b)) / (a_length*b_length)

print("同向/垂直/反向：", demo_cosine([1,0],[5,0]), demo_cosine([1,0],[0,1]), demo_cosine([1,0],[-1,0]))

同向/垂直/反向： 1.0 0.0 -1.0


## 3. 从打分走到检索：先排序，再取Top-K

Top-K是“按分数从高到低，取前K条”的简写；K是应用提供的非负整数，不是模型猜的数量。这里只找候选资料，不保证候选一定能回答问题。

先用已算好的博物馆分数观察排序。sorted是Python内置函数：`sorted(记录列表,key=取分函数,reverse=True)`返回一个新列表，不原地改变输入。

key指定按什么值排序；reverse=True表示从大到小。取分函数接收一条字典、返回它的score值。`key=museum_score`传的是函数本身，不加括号；sorted在内部把每条记录传给它调用。可以写普通def，不必先记lambda。相同分数保留原有次序，这叫稳定排序。[Python官方排序说明](https://docs.python.org/3/howto/sorting.html)。

In [16]:
museum_candidates = [
    {"id":"m2", "text":"馆内设有寄存柜。", "source":"服务指南#寄存", "score":0.2},
    {"id":"m1", "text":"入馆前需要线上预约。", "source":"参观指南#预约", "score":0.9},
    {"id":"m3", "text":"预约后凭码入馆。", "source":"参观指南#入馆", "score":0.8},
]
def museum_score(record):
    return record["score"]

ordered = sorted(museum_candidates, key=museum_score, reverse=True)
print("原来：", [record["id"] for record in museum_candidates])
print("排序：", [record["id"] for record in ordered])
print("Top-2：", [record["id"] for record in ordered[:2]])
print("Top-0：", ordered[:0])
print("Top-9实际条数：", len(ordered[:9]))

原来： ['m2', 'm1', 'm3']
排序： ['m1', 'm3', 'm2']
Top-2： ['m1', 'm3']
Top-0： []
Top-9实际条数： 3


执行顺序：先按score排成m1/m3/m2，再用[:2]取m1/m3。切片[:k]取前k项；[:0]是空列表，不是上一课的[-0:]陷阱；k超过数量只返回已有项。

变化：空列表排序再切片仍为空。同分时保留原输入顺序，而不是另外按id排。反例：先截取原来的前两条，再排序，会漏掉m3。还有一种错误是只排序分数：分数列表变了，正文与来源却没同步，结果就对不上。

真实检索先为**每条资料构造整条候选记录**，再对这些记录排序；vector留在资料库中，候选返回id/text/source/score。

In [ ]:
wrong_first_two = sorted(museum_candidates[:2], key=museum_score, reverse=True)
print("先截取的错误结果：", [record["id"] for record in wrong_first_two])
ties = [{"id":"z", "score":0.8}, {"id":"a", "score":0.8}]
print("同分稳定次序：", [record["id"] for record in sorted(ties, key=museum_score, reverse=True)])
print("空资料：", sorted([], key=museum_score, reverse=True)[:2])

### 3.1 课堂练习2：校园候选的Top-K（5分钟）

实现`campus_top(candidates,limit)->list[dict]`。输入candidates是已打分的校园记录列表，每条恰有name:str、similarity:float；limit是非负int。按similarity降序取最多limit条；同分保持输入顺序；0返回[]；不足取全部；不要修改输入。不需要计算向量或调用校园题1。

例如输入[{"name":"自习室","similarity":0.2},{"name":"借书","similarity":0.9}]、limit=1，输出[{"name":"借书","similarity":0.9}]。这里name/similarity是练习字段；课后资料改用id/text/source/score，字段不能混用。

In [17]:
def campus_top(candidates: list[dict], limit: int) -> list[dict]:
    def get_similarity(record):
        return record["similarity"]

    return sorted(
        candidates,
        key=get_similarity,
        reverse=True,
    )[:limit]

In [18]:
try:
    _check_case = "课堂练习2尚未定义campus_top"
    assert callable(globals().get("campus_top")), "先运行campus_top答题格"
    _check_rows = [{"name":"自习室","similarity":0.2},{"name":"借书","similarity":0.9},{"name":"续借","similarity":0.8}]
    _check_before = [r.copy() for r in _check_rows]
    _check_case = "输入未排序时，先排序后取2条"
    _check_actual = campus_top(_check_rows, 2)
    assert isinstance(_check_actual, list), "campus_top必须返回列表，不是None"
    assert _check_actual == [_check_rows[1],_check_rows[2]], "应返回借书、续借两条完整记录"
    assert _check_rows == _check_before, "不要原地排序输入candidates"
    _check_case = "0条、超过资料数、空资料"
    assert campus_top(_check_rows,0) == [], "limit=0返回[]"
    assert len(campus_top(_check_rows,9)) == 3, "limit过大不能补造记录"
    assert campus_top([],2) == [], "空候选应返回[]"
    _check_case = "同分保持输入顺序"
    _check_ties = [{"name":"乙","similarity":0.8},{"name":"甲","similarity":0.8}]
    assert campus_top(_check_ties,2) == _check_ties, "同分不要再按名称排序"
    print("✅ 课堂练习2通过")
except AssertionError as _check_error:
    print("❌ 课堂练习2：", str(_check_error))
except Exception:
    print("❌ 课堂练习2：", _check_case if "_check_case" in globals() else "请先运行本题答题单元", "；检查返回类型、字段及计算规则，不要改检测代码。")

✅ 课堂练习2通过


<details>
<summary>独立完成并检测后，再查看本题参考</summary>

```python
def campus_top(candidates, limit):
    def get_similarity(record):
        return record["similarity"]
    return sorted(candidates, key=get_similarity, reverse=True)[:limit]
```

它只对已有分数排序；课后search_chunks还需要把查询向量与各资料向量打分，并关联原文来源。

</details>

### 3.2 Top-K一定有候选，不代表有依据

有3条资料、K=2，即使每条都不相关，也会选出其中“相对最高”的2条。因此再设一个**阈值（门槛）**min_score：只留下score>=min_score的候选。这里“合格”仅指通过本课分数门槛，不保证事实正确。

门槛由应用或开发者设定，不是用户必须提交的API字段。0.8只用于演示，不能搬到所有模型；真实任务要用已标注的相关/不相关问题检查遗漏与误收，再调门槛。分数0.8不是80%正确率。

正常：0.9和0.8达到0.8门槛；变化：提高到0.95可以一个也没有；反例：最高只有0.3还强行留下第一条，会把无关资料当依据。

下面只演示筛选，不生成答案，也不调用模型。

In [19]:
min_score = 0.8
qualified = []
for record in ordered[:2]:
    if record["score"] >= min_score:
        qualified.append(record)
print("门槛0.8：", [r["id"] for r in qualified])
print("门槛0.95：", [r["id"] for r in ordered[:2] if r["score"] >= 0.95])
low_candidates = [{"id":"m9","score":0.3}]
print("最高分也不合格：", [r for r in low_candidates if r["score"] >= 0.8])

门槛0.8： ['m1', 'm3']
门槛0.95： []
最高分也不合格： []


### 3.3 课堂练习3：留下能过门槛的校园名称（3～5分钟）

实现`campus_evidence(candidates,min_similarity)->list[str]`。candidates是按similarity降序排好的校园候选，格式同练习2；min_similarity是float。只返回similarity>=门槛的name，保持候选顺序、不修改输入；全部不合格或空输入均返回[]。本题**返回名称字符串列表**，不是字典列表。

例：[{"name":"借书","similarity":0.9},{"name":"续借","similarity":0.8}]、门槛0.8 → ["借书","续借"]；门槛0.95 → []。只筛选，不生成校园服务答案。

In [20]:
def campus_evidence(candidates: list[dict], min_similarity: float) -> list[str]:
    # TODO：保留达到门槛的name
    names = []

    for record in candidates:
        if record["similarity"] >= min_similarity:
            names.append(record["name"])

    return names

In [21]:
try:
    _check_case = "课堂练习3尚未定义campus_evidence"
    assert callable(globals().get("campus_evidence")), "先运行campus_evidence答题格"
    _check_rows = [{"name":"借书","similarity":0.9},{"name":"续借","similarity":0.8},{"name":"自习","similarity":0.2}]
    _check_before = [r.copy() for r in _check_rows]
    _check_case = "包含恰好等于门槛的记录，返回名称列表"
    assert campus_evidence(_check_rows,0.8) == ["借书","续借"], "门槛含等号；返回name字符串，不是记录或分数"
    _check_case = "没有足够相似资料时返回空"
    assert campus_evidence(_check_rows,0.95) == [], "不能强行留下最高分"
    assert campus_evidence([],0.8) == [], "空候选返回[]"
    assert _check_rows == _check_before, "筛选不要修改输入记录"
    print("✅ 课堂练习3通过")
except AssertionError as _check_error:
    print("❌ 课堂练习3：", str(_check_error))
except Exception:
    print("❌ 课堂练习3：", _check_case if "_check_case" in globals() else "请先运行本题答题单元", "；检查返回类型、字段及计算规则，不要改检测代码。")

✅ 课堂练习3通过


<details>
<summary>独立完成并检测后，再查看本题参考</summary>

```python
def campus_evidence(candidates, min_similarity):
    names = []
    for record in candidates:
        if record["similarity"] >= min_similarity:
            names.append(record["name"])
    return names
```

注意区分输入记录和输出名称；课后filter_hits需要保留整条候选资料及其来源，不是只返回名称。

</details>

### 3.4 把T1～T4接起来：博物馆示例

下面资料的向量与查询向量全是手工坐标。仅演示：文字不同也能按事先指定的向量算分；不能宣称模型真的理解了两句话。

展示名museum_hits是示例变量；score是demo_cosine计算的float。每条资料打分后构造新字典，不把score写回资料。来源字段一直跟着同一条原文走。

In [24]:
museum_records = [
    {"id":"m1","text":"入馆前需要线上预约。","source":"参观指南#预约","vector":[1.0,0.0]},
    {"id":"m2","text":"预约后凭码入馆。","source":"参观指南#入馆","vector":[0.8,0.6]},
    {"id":"m3","text":"馆内设有寄存柜。","source":"服务指南#寄存","vector":[0.0,1.0]},
]
museum_question = "参观前要先登记吗？"
museum_query = [1.0,0.0]  # 教师给这道固定问题指定的教学向量；没有运行编码模型
museum_hits = []
for record in museum_records:
    museum_hits.append({
        "id":record["id"], "text":record["text"], "source":record["source"],
        "score":demo_cosine(museum_query,record["vector"]),
    })
museum_top_two = sorted(museum_hits,key=museum_score,reverse=True)[:2]
museum_evidence = [r for r in museum_top_two if r["score"] >= 0.9]
print("固定问题：", museum_question)
print("候选资料：", museum_evidence)
print("注意：这还不是模型生成的回答。")

固定问题： 参观前要先登记吗？
候选资料： [{'id': 'm1', 'text': '入馆前需要线上预约。', 'source': '参观指南#预约', 'score': 1.0}]
注意：这还不是模型生成的回答。


### 3.5 关键词与向量：不要把一个例子当评估结果

关键词检索利用字面词语匹配；本格用子串in作最小例子，不是完整分词或排序系统。向量检索利用编码后的相近程度，可以用于不同措辞匹配，但也可能把主题相近、并不含答案的资料排前面。编号、型号、专有名词的精确匹配常仍有价值，两者不是谁总能代替谁。

下一格用固定问题和资料文件查看差异。模型生成端没有实现，query["vector"]是教师预先指定的，并非从任意输入问题自动算出。文件的space_id是str，由教师标记这套教学坐标的版本；只写同名标签并不能自动保证真实模型兼容。chunks是资料列表，queries是固定问题列表；每条问题的text为str、vector为list[float]、relevant_ids为list[str]。relevant_ids由教师提供，是人工期望命中的资料id，用来查计算流程，不是模型预测的标签。

json是标准库；json.loads(text)把JSON文本解析成Python对象。Path.read_text(encoding="utf-8")读取文件字符串。这里只读本课静态数据，绝不读.env。

In [27]:
import json
support_data = json.loads((lesson_dir / "support_vectors.json").read_text(encoding="utf-8"))
print("向量来源：", support_data["vector_origin"])
fixed_query = support_data["queries"][0]
print("固定查询：", fixed_query["text"])
print("子串‘口令’命中：", [r["id"] for r in support_data["chunks"] if "口令" in r["text"]])
support_scores = [(r["id"], demo_cosine(fixed_query["vector"],r["vector"])) for r in support_data["chunks"]]
print("按已给坐标计算：", support_scores)
print("人工期望：", fixed_query["relevant_ids"])

向量来源： 教师手工指定的二维教学向量；不是模型Embedding或真实语义质量基准
固定查询： 口令忘了怎么恢复？
子串‘口令’命中： []
按已给坐标计算： [('s1', 1.0), ('s2', 0.8), ('s3', 0.0), ('s4', -1.0)]
人工期望： ['s1']


### 3.6 真实Embedding生成端长什么样？（只需理解）

真实系统把文档文本交给适合检索的Embedding模型，得到文档向量；查询用该模型规定的查询编码配置生成向量。它输出数字表示，不输出自然语言回答。模型应适合资料语言和查询任务；真实生成和检索质量需要单独验证。

本课没有提供一个会伪造模型结果的embed(text)函数，也没有把聊天接口随口生成的数字冒充Embedding。若以后接在线服务或本地模型，需要另行处理费用/下载、编码配置和向量版本。现在数据文件只有固定问题，不能更改问题字符串却继续用旧向量声称完成新问题检索。

向量数据库是帮助保存、查找大量向量的系统；我们只有几条资料，逐条算分即可。它不是Embedding模型，也不是本课必需依赖。

高分不证明资料真实、最新或含答案；门槛只能作为筛选策略之一。下课再做引用、无答案处理与回答校验。[官方语义检索流程](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)。

## 4. 课后作业：客服资料检索

任务背景：聊天后端回答前，先从客服资料中挑出候选。只修改vector_search.py的三个函数体，不改接口和测试。全部输入是下列有效类型，不需要通用字符串清理、类型防御、网络或数据库。

### 任务1：cosine_similarity(a,b)->float

a、b都是非空list[float]，分量为有限数。输出余弦分数，测试允许浮点舍入误差。维度不同、或任一向量全部为0时抛带有非空说明的ValueError（说明文字不固定）。除此以外输入保证有效。支持任意相同维度，不要只写死二维。

例：[2.0,0.0]、[4.0,3.0] → 约0.8；[1.0,0.0]、[-1.0,0.0] → 约-1.0。不要把点积当余弦，不要求对输入作修改。

### 任务2：search_chunks(query_vector,chunks,top_k)->list[dict]

| 参数 | 类型 | 数据由谁提供 |
|---|---|---|
| query_vector | list[float] | 应用提供的已有查询向量，本课由教师固定 |
| chunks | list[dict] | 应用提供的资料记录，字段见下方 |
| top_k | 非负int | 应用设定最多返回数量，必须显式传入 |

每条资料恰含id:str、text:str、source:str、vector:list[float]；向量与查询保证非零、同维度并属于同一套兼容教学坐标。chunks可为空。为资料计算余弦分数，返回最多top_k条、分数降序；同分保持资料输入顺序。top_k=0返回[]；超过资料数时只返回已有资料；不改变输入。

每条输出恰含id/text/source/score，前三项保留原值，score为余弦float，不带vector。这个任务不按门槛删候选，不生成客服回答。

例：query_vector=[1.0,0.0]，chunks=[{"id":"s3","text":"可导出PDF。","source":"导出手册","vector":[0.0,1.0]},{"id":"s1","text":"邮箱重置密码。","source":"账号手册","vector":[1.0,0.0]}]，top_k=1。

应返回[{"id":"s1","text":"邮箱重置密码。","source":"账号手册","score":1.0}]。

### 任务3：filter_hits(hits,min_score)->list[dict]

hits是任务2输出的候选列表，min_score是[-1,1]范围的float，由应用设定。返回score>=min_score的候选，字段和值不变，保留既有顺序，不修改输入。返回新列表；本题不要求深复制字典。全不合格或输入为空均返回[]。不计算新分数、不补造记录。

例：hits=[{"id":"s1","text":"邮箱重置密码。","source":"账号手册","score":1.0}]、min_score=0.9 → 原记录组成的新列表；min_score=1.0仍保留；最高分只有0.3而门槛0.8时返回[]。

### 测试与学习目标

共20项离线测试：余弦7项、排序检索8项、门槛4项、组合1项。全部直接检查本课计算/检索契约，通用类型边界专项0项。在本课目录运行：

```powershell
python -m pytest -q test_vector_search.py
```

分组：-k test_cosine_ / -k test_search_ / -k test_filter_ / -k test_flow_。初始全部因TODO未完成而失败是预期，不需要改测试。参考实现不放进作业文件；如果卡住，把你写的函数或失败信息发给教学助手。

## 5. 理解题：先凭记忆选字母

不要先看检测源码。每题单选，答题格填A/B/C/D；自动检测只解释答错的题。

**第1题** 两个向量都为三维，准备比较前应如何处理？

A. 维度相同就能直接比较

B. 核对编码空间兼容再比

C. 各自从小到大排序再比

D. 分别补到相同长度再比

**第2题** 非零向量方向不变，只把一方乘以正数，余弦怎样变化？

A. 余弦得分也会跟着放大

B. 余弦得分一定降到零点

C. 余弦得分始终保持不变

D. 余弦得分变成答案概率

**第3题** 输入资料未排序，要选Top-2应采用哪种行为？

A. 全部打分后降序取前两条

B. 先取原来前两条再去打分

C. 全部打分后升序取前两条

D. 只保留分数等于一的资料

**第4题** 最高候选分数0.3，门槛0.8，本课筛选应怎样处理？

A. 最高分一条也得作为依据

B. 提高分数以后再作为依据

C. 凑够规定条数再作为依据

D. 没有通过门槛就不留依据

**第5题** 本课文件中的手工教学向量，能够支持哪种结论？

A. 证实任意中文都可以检索

B. 只验计算流程不证实语义

C. 证实真实模型能正确编码

D. 证实检索得分就是正确率

In [28]:
answers = {"q1": "B", "q2": "C", "q3": "A", "q4": "D", "q5": "B"}

In [29]:
try:
    assert isinstance(globals().get("answers"), dict), "请先运行answers答题格"
    _check_key = {"q1":"B","q2":"C","q3":"A","q4":"D","q5":"B"}
    _check_explanations = {"q1":"同维度只是必要条件；查询和资料的编码空间及配置必须兼容。","q2":"余弦除以两边长度，正比例放大不改变方向分数。","q3":"先给全部候选打分并降序排序，再取前K；提前截断会漏掉更相关资料。","q4":"门槛后可以为空；Top-K只是相对排序，不保证资料能作依据。","q5":"手工向量只验证计算和流程，真实语义质量与编码模型要另外评估。"}
    _check_passed = True
    for _check_q, _check_correct in _check_key.items():
        _check_answer = answers.get(_check_q, "")
        if not isinstance(_check_answer, str) or _check_answer.strip().upper() not in ("A","B","C","D"):
            print("❌", _check_q, "尚未作答或不是单个A/B/C/D")
            _check_passed = False
        elif _check_answer.strip().upper() != _check_correct:
            print("❌", _check_q, _check_explanations[_check_q])
            _check_passed = False
    if _check_passed:
        print("✅ 理解题5题全部正确")
except AssertionError as _check_error:
    print("❌ 理解题：", str(_check_error))
except Exception:
    print("❌ 理解题：检查answers是否为字典，不要修改检测代码")

✅ 理解题5题全部正确


## 6. 第二天再做：隔天回忆

今天完成后先停止；**第二天**关掉参考，在独立草稿复制三个函数接口、清空实现，凭记忆重写并运行原测试；不要清空正式作业。把理解题答案在草稿清空，重答一次。3～7天后再用另一组向量复习；当前检测通过不代表长期记住。

先画出“建库”和“查询”的两条线，再解释：文本和向量谁生成谁、T2～T4谁负责、来源在排序后为什么不能丢。交替检索上一课：流式JSON仅收到半段能否完整校验？正常完成后先做哪两步？只在技能复习时交替，不把新概念混着讲。

有卡点只回看对应节，不需要从头重读。第04课recipe_recent的保留全部分支仍需你补正：用新列表并复制每条字典；本课没有覆盖旧文件。

[Markdown速查](速查_向量检索.md) / [HTML速查](../reference/0002-vector-retrieval.html)。新术语尚未有你的独立作答证据，不提前写入“已理解”术语表。

## 7. 原始资料与提问

核对日期：2026-10-14。以下是用于本课概念与公式的原始资料，不要求安装页面里的大型库或运行联网示例：

- [Google：Embedding空间](https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space)：表示与维度、坐标不可直接解读。
- [Sentence Transformers：语义检索](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)：资料与查询编码、计算相似度、选候选。
- [scikit-learn：余弦公式](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)：归一化点积；本课零向量策略自行明确，不照搬该库约定。
- [Python：sqrt](https://docs.python.org/3/library/math.html#math.sqrt)、[排序](https://docs.python.org/3/howto/sorting.html)：参数、降序与稳定性。

有疑问，把具体章节、代码或检测提示发给教学助手继续解释；不用猜字段或库函数的含义。